Model Training and Evaluation: Adversarial Phishing DetectionThis notebook is dedicated to training and evaluating the final Machine Learning (ML) and Deep Learning (DL) models for the phishing detection system. We separate this step from feature engineering to maintain a clean, modular, and reproducible workflow.1. Data Preparation and SplittingThe primary goal of this step is to load the dataset containing all the engineered features (created in the project_overview.ipynb and feature_engineer.py) and split it into training and testing sets.The input data is assumed to be stored as ../processed/sessions_engineered.csv.1.1 Loading DataWe load the data, define the features X and the target label X, and then perform an 80/20 train-test split, ensuring the split is stratified to maintain the original class distribution in both sets.

MLP Deep learning model

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import random
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = "../../processed/Feature.csv"  
df = pd.read_csv(DATA_PATH)
df = df.dropna(axis=0)

label_candidates = ['label', 'Label', 'target', 'Target', 'y']
label_col = next((c for c in label_candidates if c in df.columns), df.columns[-1])

X = df.drop(columns=[label_col])
y = df[label_col]

if y.dtype == object or not np.issubdtype(y.dtype, np.number):
    le = LabelEncoder()
    y = le.fit_transform(y)

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

X_processed = preprocessor.fit_transform(X)


kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
accuracies, precisions, recalls, f1s = [], [], [], []

for train_idx, test_idx in kf.split(X_processed, y):
    X_train, X_test = X_processed[train_idx], X_processed[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    
    classes = np.unique(y_train)
    class_weights_values = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weights = dict(zip(classes, class_weights_values))

    
    input_dim = X_train.shape[1]
    model = Sequential([
        Dense(32, input_dim=input_dim, activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(16, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.4),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    
    early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

    
    model.fit(X_train, y_train,
              validation_split=0.2,
              epochs=50,
              batch_size=16,
              verbose=0,
              callbacks=[early_stop],
              class_weight=class_weights)

    
    y_pred = (model.predict(X_test) > 0.5).astype(int)
    accuracies.append(accuracy_score(y_test, y_pred))
    precisions.append(precision_score(y_test, y_pred))
    recalls.append(recall_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred))


print("=== Optimized Stable MLP Metrics (5-Fold CV) ===")
print(f"Accuracy: {np.mean(accuracies):.4f}")
print(f"Precision: {np.mean(precisions):.4f}")
print(f"Recall: {np.mean(recalls):.4f}")
print(f"F1-score: {np.mean(f1s):.4f}")
model.save("mlp_stable_model.keras")  



joblib.dump(preprocessor, "mlp_preprocessor.pkl")

print("\nModel and preprocessor saved successfully!")
model = load_model("mlp_stable_model.keras")
preprocessor = joblib.load("mlp_preprocessor.pkl")


TEST_PATH = "../../processed/test_mlp.csv"  
test_df = pd.read_csv(TEST_PATH)

if 'label' in test_df.columns:
    y_test = test_df['label']
    X_test = test_df.drop(columns=['label'])
else:
    y_test = None
    X_test = test_df.copy()


train_cols = X.columns.tolist()

missing = [c for c in train_cols if c not in X_test.columns]
for c in missing:
    if c in numeric_cols:
        X_test[c] = 0
    else:
        X_test[c] = ""  


extra = [c for c in X_test.columns if c not in train_cols]
if extra:
    X_test = X_test.drop(columns=extra)


X_test = X_test[train_cols]

X_test_processed = preprocessor.transform(X_test)


y_pred = (model.predict(X_test_processed) > 0.5).astype(int)


if y_test is not None:
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("\n=== Test Data Metrics ===")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
else:
    print("\nPredictions on test data:")
    print(y_pred)


c:\Python313\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 607ms/step


c:\Python313\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step


c:\Python313\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 563ms/step


c:\Python313\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 541ms/step


c:\Python313\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
=== Optimized Stable MLP Metrics (5-Fold CV) ===
Accuracy: 0.9000
Precision: 1.0000
Recall: 0.9000
F1-score: 0.9333

Model and preprocessor saved successfully!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step

=== Test Data Metrics ===
Accuracy: 0.5000
Precision: 0.5000
Recall: 1.0000
F1-score: 0.6667
